# HRV cardiac-arrest model — training & evaluation only

Assumes you already have a features CSV (e.g. `hrv_features_dataset.csv` from the
data-extraction notebook) with:
- one row per 5-minute window
- HRV feature columns (prefixed `HRV_...`)
- a `record_id` column (patient/recording identifier, used for grouped splitting)
- a `label` column (0 = normal, 1 = pre-arrest)

Upload your CSV to the Colab session (folder icon on the left → upload), then set
`CSV_PATH` below.

## 1. Setup

In [ ]:
!pip install scikit-learn xgboost imbalanced-learn pandas numpy matplotlib seaborn tabulate -q

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (confusion_matrix, classification_report, accuracy_score,
                              precision_score, recall_score, f1_score, roc_auc_score,
                              average_precision_score, precision_recall_curve)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load the dataset

In [ ]:
CSV_PATH = '/content/hrv_features_dataset.csv'  # <-- change if your file is named/located differently

full_df = pd.read_csv(CSV_PATH)

meta_cols = ['record_id', 'window_end_sec', 'label']
feature_cols = [c for c in full_df.columns if c not in meta_cols]

print("Dataset shape:", full_df.shape)
print("Class balance:\n", full_df['label'].value_counts())
print("Unique patients:", full_df['record_id'].nunique())

## 3. Clean features

Drop any HRV columns that are mostly missing (some metrics fail to compute on
short/irregular windows) and drop rows still missing values in the columns you keep.

In [ ]:
nan_frac = full_df[feature_cols].isna().mean()
keep_cols = nan_frac[nan_frac < 0.2].index.tolist()
full_df = full_df.dropna(subset=keep_cols)

print(f"Using {len(keep_cols)} of {len(feature_cols)} features")
print("Dataset shape after cleaning:", full_df.shape)

## 4. Train/test split — grouped by patient (no leakage), on the REAL (unaugmented) data

In [ ]:
X = full_df[keep_cols]
y = full_df['label'].astype(int)
groups = full_df['record_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # test set stays 100% real, untouched by any augmentation

print("Train windows (real):", X_train.shape, " Test windows (real, untouched):", X_test.shape)
print("Train patients:", groups_train.nunique(), " Test patients:", groups.iloc[test_idx].nunique())
print("Train class balance:\n", y_train.value_counts())

## 5. Augment the training set with SMOTE (synthetic minority oversampling)

**Why SMOTE instead of just repeating rows:** duplicating the same 355 pre-arrest
windows teaches the model nothing new — it just sees identical points more often,
which increases overfitting risk. SMOTE instead generates *synthetic* pre-arrest
examples by interpolating between nearby real pre-arrest windows in feature
space — new (if artificial) data points along the same physiological pattern,
not exact copies.

**Why this only touches the training set:** the test set above was already split
off from real, natural data before this cell runs. SMOTE is applied only to
`X_train_scaled`/`y_train` — the test set is never touched, so every metric you
compute later still reflects genuine, unaugmented data. Synthesizing fake
samples into your evaluation set would make any resulting metric meaningless.

`k_neighbors` controls how many real neighbors SMOTE interpolates between —
lowered to 5 here since you only have a few hundred real positive training
examples to draw from; the default (5) is usually fine, but if this cell errors
about too few neighbors, lower it further (e.g. 3).

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f"Before SMOTE — Normal: {sum(y_train==0)}, Pre-arrest: {sum(y_train==1)}")
print(f"After SMOTE  — Normal: {sum(y_train_smote==0)}, Pre-arrest: {sum(y_train_smote==1)}")

# groups_train doesn't map cleanly onto the synthetic rows (they don't belong to
# a real patient), so CV in the next section uses a plain StratifiedKFold on the
# SMOTE-augmented data rather than StratifiedGroupKFold.

## 6. Model comparison (cross-validation on SMOTE-augmented training data)

### Why classical ML over deep learning here
Even with SMOTE-generated synthetic examples, the underlying number of real
*patients* hasn't grown — you still only have as many independent people as
your real dataset contains. Deep learning models (LSTM/CNN on raw R-R
sequences) need hundreds to thousands of independent subjects to generalize
rather than memorize; SMOTE doesn't manufacture new patients, just new points
along the existing ones' feature patterns. Engineered HRV features + tree-based
models (Random Forest, XGBoost) remain the better fit for this data regime.

### Why plain StratifiedKFold here, not StratifiedGroupKFold
The synthetic SMOTE rows are interpolated between real patients and don't
belong to any single real `record_id`, so patient-grouped CV doesn't cleanly
apply anymore. This CV step is only used to rank/compare the 4 candidate
models against each other — the number it reports is a less strict internal
comparison, not your real generalization estimate. Section 9's held-out test
set (100% real, ungrouped-during-SMOTE data) is what tells you how the chosen
model actually performs on unseen patients.

In [ ]:
from sklearn.model_selection import StratifiedKFold

scale_pos_weight = 1.0  # already balanced by SMOTE, so no extra reweighting needed
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

candidate_models = {
    "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "RandomForest":       RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    "SVM_RBF":            SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
    "XGBoost":            XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                         eval_metric='logloss', random_state=RANDOM_STATE),
}

cv_results = {}

for name, model in candidate_models.items():
    ap_scores, f1s = [], []
    for tr_i, val_i in skf.split(X_train_smote, y_train_smote):
        model.fit(X_train_smote[tr_i], y_train_smote[tr_i])
        proba = model.predict_proba(X_train_smote[val_i])[:, 1]
        preds = model.predict(X_train_smote[val_i])
        ap_scores.append(average_precision_score(y_train_smote[val_i], proba))
        f1s.append(f1_score(y_train_smote[val_i], preds))
    cv_results[name] = {"mean_pr_auc": np.mean(ap_scores), "mean_f1": np.mean(f1s)}
    print(f"{name}: PR-AUC={np.mean(ap_scores):.3f}  F1={np.mean(f1s):.3f}")

cv_results_df = pd.DataFrame(cv_results).T.sort_values('mean_pr_auc', ascending=False)
cv_results_df

## 7. Select the best model and retrain on the full SMOTE-augmented training set

In [ ]:
best_model_name = cv_results_df.index[0]
print("Selected model:", best_model_name)

best_model = candidate_models[best_model_name]
best_model.fit(X_train_smote, y_train_smote)

## 8. Pick a decision threshold (don't use the default 0.5)

With a rare positive class, the default 0.5 threshold usually collapses to
"always predict normal" — high accuracy, zero precision/recall/F1. Instead, pick
a threshold off the precision-recall curve based on the tradeoff you want. For an
early-warning system, recall usually matters more than precision (missing a real
pre-arrest window is worse than an extra false alarm) — adjust `TARGET_RECALL`
to fit what you're comfortable with.

In [ ]:
y_proba_test = best_model.predict_proba(X_test_scaled)[:, 1]
pr_auc_test = average_precision_score(y_test, y_proba_test)

precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_test)

plt.figure(figsize=(6, 5))
plt.plot(recalls, precisions)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision-Recall curve — {best_model_name} (PR-AUC = {pr_auc_test:.3f})")
plt.tight_layout()
plt.savefig('/content/precision_recall_curve.png', dpi=150)
plt.show()

TARGET_RECALL = 0.7   # <-- adjust this to trade off precision vs recall
idx = np.argmin(np.abs(recalls[:-1] - TARGET_RECALL))  # thresholds has one fewer element than recalls/precisions
chosen_threshold = thresholds[idx]
print(f"At recall={recalls[idx]:.2f}, precision={precisions[idx]:.2f}, threshold={chosen_threshold:.3f}")

## 9. Held-out test set evaluation — default threshold AND tuned threshold

In [ ]:
# --- default 0.5 threshold, for reference ---
y_pred_default = (y_proba_test >= 0.5).astype(int)
print("=== Default threshold (0.5) ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_default):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_default, zero_division=0):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_default, zero_division=0):.3f}")
print(f"F1 score:  {f1_score(y_test, y_pred_default, zero_division=0):.3f}")

# --- tuned threshold from the precision-recall curve ---
y_pred = (y_proba_test >= chosen_threshold).astype(int)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_proba_test)

print(f"\n=== Tuned threshold ({chosen_threshold:.3f}) ===")
print(f"Accuracy:  {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall:    {rec:.3f}")
print(f"F1 score:  {f1:.3f}")
print(f"ROC-AUC:   {auc:.3f}")
print(f"PR-AUC:    {pr_auc_test:.3f}")
print("\nFull classification report (tuned threshold):\n",
      classification_report(y_test, y_pred, target_names=["Normal", "Pre-arrest"], zero_division=0))

## 10. Confusion matrix (tuned threshold) — best model

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Normal", "Pre-arrest"], yticklabels=["Normal", "Pre-arrest"])
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title(f"Confusion matrix — {best_model_name} (threshold={chosen_threshold:.2f})")
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()

## 10b. Confusion matrix for ALL 4 models (side by side)

Each model gets its own tuned threshold (same target-recall logic as the best
model above) so the comparison is fair — a model isn't penalized just because
0.5 happens to be a bad cutoff for it. This lets you see precisely how each
model trades off false positives against false negatives, not just which one
"won" on PR-AUC.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
all_model_metrics = {}

for ax, (name, model) in zip(axes.flatten(), candidate_models.items()):
    model.fit(X_train_smote, y_train_smote)  # retrain on SMOTE-augmented training set
    proba = model.predict_proba(X_test_scaled)[:, 1]  # evaluated on the real, untouched test set

    p, r, t = precision_recall_curve(y_test, proba)
    idx_m = np.argmin(np.abs(r[:-1] - TARGET_RECALL))
    thresh_m = t[idx_m]
    pred_m = (proba >= thresh_m).astype(int)

    acc_m = accuracy_score(y_test, pred_m)
    prec_m = precision_score(y_test, pred_m, zero_division=0)
    rec_m = recall_score(y_test, pred_m, zero_division=0)
    f1_m = f1_score(y_test, pred_m, zero_division=0)
    auc_m = roc_auc_score(y_test, proba)
    prauc_m = average_precision_score(y_test, proba)

    all_model_metrics[name] = {
        "threshold": thresh_m, "accuracy": acc_m, "precision": prec_m,
        "recall": rec_m, "f1": f1_m, "roc_auc": auc_m, "pr_auc": prauc_m
    }

    cm_m = confusion_matrix(y_test, pred_m)
    sns.heatmap(cm_m, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=["Normal", "Pre-arrest"], yticklabels=["Normal", "Pre-arrest"])
    ax.set_title(f"{name}\nthresh={thresh_m:.3f}  P={prec_m:.2f} R={rec_m:.2f} F1={f1_m:.2f}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.tight_layout()
plt.savefig('/content/confusion_matrix_all_models.png', dpi=150)
plt.show()

all_metrics_df = pd.DataFrame(all_model_metrics).T.sort_values('pr_auc', ascending=False)
print(all_metrics_df.round(3))

## 11. Feature importance

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=keep_cols).sort_values(ascending=False)
    top_features = importances.head(15)

    plt.figure(figsize=(7, 5))
    sns.barplot(x=top_features.values, y=top_features.index, color="#1D9E75")
    plt.xlabel("Importance")
    plt.title(f"Top 15 features — {best_model_name}")
    plt.tight_layout()
    plt.savefig('/content/feature_importance.png', dpi=150)
    plt.show()
else:
    top_features = pd.Series(dtype=float)
    print("Selected model has no feature_importances_ (e.g. Logistic/SVM) — "
          "inspect model.coef_ instead if you want feature-level insight.")

## 12. Auto-generated report

Run this last cell and paste the printed markdown back into chat — gives full
context (dataset stats, model comparison, chosen threshold, final metrics,
confusion matrix, top features) needed to keep improving the model together.

In [ ]:
report_lines = []
report_lines.append("# HRV cardiac-arrest-warning model — training report\n")

report_lines.append("## Dataset")
report_lines.append(f"- Total real labeled windows: {len(full_df)} (Normal={sum(full_df['label']==0)}, Pre-arrest={sum(full_df['label']==1)})")
report_lines.append(f"- Train patients: {groups_train.nunique()}, Test patients: {groups.iloc[test_idx].nunique()}")
report_lines.append(f"- Real training windows before SMOTE — Normal: {sum(y_train==0)}, Pre-arrest: {sum(y_train==1)}")
report_lines.append(f"- Training windows after SMOTE — Normal: {sum(y_train_smote==0)}, Pre-arrest: {sum(y_train_smote==1)}")
report_lines.append(f"- Test set: 100% real, unaugmented ({len(y_test)} windows)")
report_lines.append(f"- Feature count: {len(keep_cols)}")
report_lines.append(f"- Feature list: {', '.join(keep_cols)}\n")

report_lines.append("## Model comparison (5-fold StratifiedKFold CV on SMOTE-augmented data, ranked by PR-AUC)")
report_lines.append(cv_results_df.to_markdown())
report_lines.append(f"\n**Selected model: {best_model_name}**\n")

report_lines.append("## Decision threshold")
report_lines.append(f"- Target recall: {TARGET_RECALL}")
report_lines.append(f"- Chosen threshold: {chosen_threshold:.3f} (probability >= this = predicted pre-arrest)\n")

report_lines.append("## Held-out test set performance (tuned threshold)")
report_lines.append(f"- Accuracy:  {acc:.3f}")
report_lines.append(f"- Precision: {prec:.3f}")
report_lines.append(f"- Recall:    {rec:.3f}")
report_lines.append(f"- F1 score:  {f1:.3f}")
report_lines.append(f"- ROC-AUC:   {auc:.3f}")
report_lines.append(f"- PR-AUC:    {pr_auc_test:.3f}\n")

report_lines.append("## All-model comparison on held-out test set (each at its own tuned threshold)")
report_lines.append(all_metrics_df.round(3).to_markdown())
report_lines.append("")

report_lines.append("## Confusion matrix — best model (rows=true, cols=predicted; [Normal, Pre-arrest])")
report_lines.append(f"```\n{cm}\n```\n")

if len(top_features):
    report_lines.append("## Top 10 features by importance")
    for feat, val in top_features.head(10).items():
        report_lines.append(f"- {feat}: {val:.4f}")

report_text = "\n".join(report_lines)
print(report_text)

with open('/content/model_report.md', 'w') as f:
    f.write(report_text)